<https://github.com/algebraic-solving/msolve/tree/master/input_files>

Generate traces out of pcode symbolic executor

In [ ]:
import lark


grammar = r"""

file : rewrite*
rewrite : "("  "rewrite"  term term ")"
?term : "(" term* ")" -> list 
      | atom 
atom : /[^\s()]+/

%import common.WS
%ignore WS
"""
parser = lark.Lark(grammar, start='file', parser='lalr')

tree = parser.parse(
"""
    (rewrite (+ a b) (+ b a))
    (rewrite (+ (+ a b) c) (+ a (+ b c)))
    (rewrite (+ a 0) a)
""")

from z3 import *
def interp_term(t):
    match t:
        case lark.Tree("list", terms):
            return [interp_term(term) for term in terms]
        case lark.Tree("atom", [atom]):
            return atom.value
        case _:
            raise ValueError(f"Unexpected term structure: {t}")

def interp(t):
    res = []
    match t:
        case lark.Tree("file", rewrites):
            for rewrite in rewrites:
                match rewrite:
                    case lark.Tree("rewrite", [lhs, rhs]):
                        lhs_interp = interp_term(lhs)
                        rhs_interp = interp_term(rhs)
                        res.append((lhs_interp, rhs_interp))
                    case _:
                        raise ValueError(f"Unexpected rewrite structure: {rewrite}")
        case _:
            raise ValueError(f"Unexpected tree structure: {t}")
    return res
interp(tree)



[(['+', 'a', 'b'], ['+', 'b', 'a']),
 (['+', ['+', 'a', 'b'], 'c'], ['+', 'a', ['+', 'b', 'c']]),
 (['+', 'a', '0'], 'a')]

In [ ]:
%%file /tmp/group.egg

(sort G)
(function mul (G G) G)
(function inv (G) G)
(function id () G)

(rewrite (mul (id) x) x)
(rewrite (mul x (id)) x)
(rewrite (mul x (inv x)) (id))
(rewrite (mul (inv x) x) (id))
(rewrite (mul x (mul y z)) (mul (mul x y) z))
(rewrite (mul (mul x y) z) (mul x (mul y z)))


Writing /tmp/group.egg
